In [1]:
#Importações

import pandas as pd # importação do pandas
from sklearn.tree import DecisionTreeClassifier # Importação do classificador Decision Tree
from sklearn.model_selection import train_test_split # Importação das funções de treino e teste
from sklearn import metrics #Importação de módulo para aplicação de métricas de acurácia do scikit-learn
from sklearn import tree
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.metrics import RocCurveDisplay

In [5]:
import matplotlib.pyplot as plt
import graphviz
from sklearn.tree import export_graphviz
from six import StringIO
from IPython.display import Image
import pydotplus

In [6]:
pd.set_option("display.max_rows", 999)
pd.set_option("display.max_columns", 999)

In [ ]:
srag = pd.read_csv('/content/drive/MyDrive/Health Data Science/ID3 V5.0/srag_dataset_2013_a_2023-attr_set_2.csv')
srag = srag.dropna()
srag.head()

In [8]:
srag.columns

Index(['PERD_OLFT', 'PERD_PALA', 'FEBRE', 'DISPINEIA', 'SATURACAO', 'FADIGA',
       'SUPORT_VEN', 'RAIOX_RES', 'TOSSE', 'TOMO_RES', 'UTI', 'GARGANTA',
       'CS_SEXO', 'DIST_PRI_INTERN', 'TEMPO_UTI', 'POSSUI_MORBIDADE',
       'FAIXA_ETARIA', 'FAIXA_ETARIA_2', 'IMAGEM_RES', 'VACINAS', 'ANO',
       'EVOLUCAO', 'CRITERIO', 'CLASSI_FIN', 'id'],
      dtype='object')

In [9]:
srag = srag.rename(columns={'CLASSI_FIN': 'label'})

In [10]:
##Dividindo as features

feature_cols = ['PERD_OLFT', 'PERD_PALA', 'SATURACAO',
       'SUPORT_VEN', 'RAIOX_RES', 'TOSSE', 'TOMO_RES', 'UTI',
       'DIST_PRI_INTERN', 'POSSUI_MORBIDADE',
       'FAIXA_ETARIA', 'FAIXA_ETARIA_2', 'VACINAS',
       'EVOLUCAO','ANO']

In [11]:
x = srag[feature_cols]
y = srag.label


In [14]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=1, shuffle=True)

In [15]:
# Criando objeto do Classificador
clf = DecisionTreeClassifier(criterion='entropy', max_depth=4, min_samples_split=2, splitter='best')

# Treinando classificador
clf = clf.fit(x_train,y_train)

# Predizendo as classes presentes no teste
y_pred = clf.predict(x_test)

# calculando pontuação para cada predição no teste
y_score = clf.fit(x_train, y_train).predict_proba(x_test)

In [16]:
print("Accuracy:",metrics.accuracy_score(y_test, y_pred))

Accuracy: 0.910002943171203


In [ ]:
#Obtendo Métricas De Validação
tree.plot_tree(clf, feature_names=feature_cols)
plt.savefig('srag_tree.png')


In [ ]:
dot_data = StringIO()
export_graphviz(clf, out_file=dot_data,
                filled=True, rounded=True,
                special_characters=True,feature_names = feature_cols,class_names=['0','1'])
graph = pydotplus.graph_from_dot_data(dot_data.getvalue())
graph.write_png('srag.png')
Image(graph.create_png())

In [ ]:
ConfusionMatrixDisplay.from_predictions(y_test, y_pred)
plt.savefig('srag_confusion_matrix.pdf')

In [ ]:
RocCurveDisplay.from_predictions(y_test, y_score[:, 1])
plt.savefig('srag_roc_curve.pdf')


In [ ]:
feat_importance = pd.DataFrame(clf.feature_importances_, index=feature_cols, columns=['importance'])
feat_importance.sort_values(by='importance', ascending=False, inplace=True)
feat_importance.plot(kind='bar')